# ClinicalTrials.gov - Databricks Lakeflow Pipeline with Auto Loader & CDC

This pipeline pulls data from the ClinicalTrials.gov API and processes it through Bronze → Silver → Gold layers using Auto Loader streaming and Change Data Capture (CDC).

## Pipeline Architecture
- **API → Volume**: Fetches data from API and writes to Unity Catalog volume as JSON files (append-only)
- **Bronze (Auto Loader Streaming)**: Auto Loader continuously monitors volume for new files, ingests as VARIANT data type
- **Silver (CDC with SCD Type 1)**: Streams from bronze, deduplicates and tracks changes based on `nct_id`
- **Gold (Batch)**: Curated aggregated tables optimized for dashboard analytics

## True Streaming Architecture with Auto Loader
- API data is written to `/Volumes/pavan_naidu/json/raw_data/clinical_trials/` with timestamp-based filenames
- Each pipeline run appends new JSON files to the volume (never overwrites)
- Auto Loader continuously monitors the volume and automatically ingests new files
- Bronze table is truly append-only streaming table
- Silver CDC processes incremental updates and handles deduplication by nct_id

## CDC Behavior
- **Primary Key**: `nct_id` (NCT Number - unique identifier for each clinical trial)
- **Sequence By**: `sequence_number` (timestamp-based sequence to determine record order)
- **SCD Type 1**: Latest record overwrites previous record (no history tracking)
- When a clinical trial is updated (e.g., status changes from RECRUITING to COMPLETED), the silver table automatically updates the record

## Parameters (configured via webapp):
- `condition`: Medical condition to search for
- `study_type`: Type of clinical study (note: API v2 has limited filter support)
- `status`: Study recruitment status (note: API v2 has limited filter support)
- `page_size`: Number of records per API call
- `max_pages`: Maximum number of API pages to fetch

In [ ]:
import dlt
import requests
from pyspark.sql.functions import *
from pyspark.sql.types import *
import uuid

# Pipeline parameters (these will be passed from the webapp)
condition = spark.conf.get("pipeline.condition", "diabetes")
study_type = spark.conf.get("pipeline.study_type", "INTERVENTIONAL")
status = spark.conf.get("pipeline.status", "RECRUITING")
page_size = spark.conf.get("pipeline.page_size", "100")
max_pages = spark.conf.get("pipeline.max_pages", "0")  # 0 = unlimited, >0 = limit to that number

print(f"Pipeline Parameters:")
print(f"  Condition: {condition}")
print(f"  Study Type: {study_type}")
print(f"  Status: {status}")
print(f"  Page Size: {page_size}")
print(f"  Max Pages: {max_pages} ({'unlimited' if max_pages == '0' else 'limited'})")

## Helper Functions

In [ ]:
def fetch_clinical_trials_data(condition, study_type, status, page_size, max_pages):
    """
    Fetch data from ClinicalTrials.gov API v2 with parameterized inputs
    API Docs: https://clinicaltrials.gov/data-api/api
    Note: API v2 has limited filter support, using basic query parameters
    
    Parameters:
    - max_pages: 0 = fetch all pages (unlimited), >0 = limit to that number of pages
    """
    base_url = "https://clinicaltrials.gov/api/v2/studies"
    
    all_studies = []
    next_page_token = None
    page_num = 0
    max_pages_int = int(max_pages)
    
    # Determine if unlimited (0) or limited (>0)
    is_unlimited = (max_pages_int == 0)
    
    while True:
        page_num += 1
        
        # Check if we've reached the limit (only if not unlimited)
        if not is_unlimited and page_num > max_pages_int:
            print(f"Reached max_pages limit of {max_pages_int}")
            break
        
        # Build query parameters - API v2 only supports basic query parameters
        params = {
            "query.cond": condition,
            "pageSize": int(page_size)
        }
        
        # Add page token for pagination
        if next_page_token:
            params["pageToken"] = next_page_token
        
        try:
            print(f"Fetching page {page_num} from ClinicalTrials.gov API... {'(unlimited mode)' if is_unlimited else f'(limit: {max_pages_int})'}")
            print(f"Parameters: {params}")
            
            response = requests.get(base_url, params=params, timeout=60)
            response.raise_for_status()
            data = response.json()
            
            print(f"Response status: {response.status_code}")
            print(f"Response keys: {data.keys() if data else 'None'}")
            
            if "studies" in data and data["studies"]:
                all_studies.extend(data["studies"])
                print(f"Retrieved {len(data['studies'])} studies (total: {len(all_studies)})")
            else:
                print(f"No studies found in response")
                break
            
            # Check if there are more pages
            if "nextPageToken" in data and data["nextPageToken"]:
                next_page_token = data["nextPageToken"]
            else:
                print("No more pages available - reached end of data")
                break
                
        except requests.exceptions.RequestException as e:
            print(f"API Request Error on page {page_num}: {str(e)}")
            if hasattr(e, 'response') and e.response is not None:
                print(f"Response content: {e.response.text[:500]}")
            break
        except Exception as e:
            print(f"Unexpected error on page {page_num}: {str(e)}")
            break
    
    print(f"Total studies fetched: {len(all_studies)} across {page_num} pages")
    return all_studies

## Bronze Layer - Auto Loader Streaming from Volume

Bronze layer downloads API data to volume, then uses Auto Loader to stream from volume.
API download happens inside the DLT function to ensure proper execution order.
Files are written with timestamp-based names (append-only), Auto Loader detects and processes them.

In [ ]:
import json
from datetime import datetime
import os

# Volume path configuration  
CATALOG = spark.conf.get("catalog", "pavan_naidu")
SCHEMA = spark.conf.get("schema", "json")
VOLUME = spark.conf.get("volume", "raw_data")
VOLUME_PATH = f"/Volumes/{CATALOG}/{SCHEMA}/{VOLUME}/clinical_trials"

# Bronze streaming table using Auto Loader
@dlt.table(
    name="clinical_trials_bronze_autoloader",
    comment="Bronze streaming table: Downloads API data to volume with unique pull_id, then Auto Loader ingests with VARIANT type",
    table_properties={
        "quality": "bronze",
        "pipelines.autoOptimize.zOrderCols": "ingestion_timestamp",
        "delta.feature.variantType-preview": "supported",
        "delta.enableChangeDataFeed": "true"
    },
    partition_cols=["ingestion_date"]
)
def clinical_trials_bronze_autoloader():
    """
    Bronze streaming table with integrated API download:
    1. Downloads data from API and writes to volume as JSON files with unique pull_id
    2. Uses Auto Loader to stream from volume
    3. Converts raw_data to variant_data and drops raw_data (no duplication)
    True streaming architecture - files are appended to volume, never overwritten
    Each API pull gets a unique pull_id (UUID) to track data lineage
    No filtering in bronze - all records pass through
    Data is written in batches to avoid Spark RPC size limits
    """
    
    # Step 1: Download API data and write to volume
    def write_api_data_to_volume():
        """Download data from API and write to volume as JSON files with unique pull_id"""
        # Generate unique pull_id for this API fetch
        pull_id = str(uuid.uuid4())
        print(f"Generated pull_id: {pull_id}")
        
        # Fetch data from API
        studies_data = fetch_clinical_trials_data(
            condition=condition,
            study_type=study_type,
            status=status,
            page_size=page_size,
            max_pages=max_pages
        )
        
        if not studies_data:
            print("WARNING: No data fetched from API")
            return 0, pull_id
        
        # Create timestamp-based filename (append-only)
        timestamp = datetime.now().strftime("%Y%m%d_%H%M%S_%f")
        
        print(f"Writing {len(studies_data)} studies to volume with pull_id: {pull_id}")
        
        # Create volume directory if needed
        try:
            dbutils.fs.mkdirs(VOLUME_PATH)
        except Exception as e:
            print(f"Volume path exists or error: {e}")
        
        # Write data in batches to avoid RPC size limits
        batch_size = 500  # Write 500 records per batch
        total_written = 0
        
        for batch_num, i in enumerate(range(0, len(studies_data), batch_size), 1):
            batch = studies_data[i:i + batch_size]
            
            # Prepare records with metadata - store as raw_data for compatibility
            records = []
            for study in batch:
                record = {
                    "raw_data": json.dumps(study),  # Store as JSON string for parse_json
                    "study_id": study.get("protocolSection", {}).get("identificationModule", {}).get("nctId", "UNKNOWN"),
                    "pull_id": pull_id,  # Unique ID for this API pull
                    "ingestion_timestamp": datetime.now().isoformat(),
                    "ingestion_date": datetime.now().date().isoformat(),
                    "source": "clinicaltrials.gov",
                    "pipeline_condition": condition,
                    "pipeline_study_type": study_type,
                    "pipeline_status": status
                }
                records.append(record)
            
            # Write this batch to volume as JSON
            batch_filename = f"clinical_trials_{condition}_{timestamp}_batch{batch_num}.json"
            batch_file_path = f"{VOLUME_PATH}/{batch_filename}"
            
            temp_df = spark.createDataFrame(records)
            temp_df.coalesce(1).write.mode("append").json(batch_file_path)
            
            total_written += len(records)
            print(f"Batch {batch_num}: Wrote {len(records)} records to {batch_filename} (total: {total_written}/{len(studies_data)})")
        
        print(f"Successfully wrote all {total_written} records to volume with pull_id: {pull_id}")
        return total_written, pull_id
    
    # Execute API download
    records_written, pull_id = write_api_data_to_volume()
    print(f"API fetch completed. {records_written} records written to volume with pull_id: {pull_id}")
    
    # Step 2: Stream from volume using Auto Loader
    df = (
        spark.readStream
        .format("cloudFiles")
        .option("cloudFiles.format", "json")
        .option("cloudFiles.schemaLocation", f"{VOLUME_PATH}/_schema")
        .option("cloudFiles.inferColumnTypes", "true")
        .option("cloudFiles.useNotifications", "false")
        .load(VOLUME_PATH)
    )
    
    # Add sequence number for CDC ordering
    from pyspark.sql.functions import unix_timestamp, to_timestamp
    df = df.withColumn("ingestion_timestamp_ts", to_timestamp(col("ingestion_timestamp")))
    df = df.withColumn("sequence_number", unix_timestamp(col("ingestion_timestamp_ts")))
    
    # Parse JSON as VARIANT type
    df = df.withColumn("variant_data", parse_json(col("raw_data")))
    
    # Drop raw_data since we only need variant_data in bronze table
    df = df.drop("raw_data")
    
    # No filtering - all records pass through to silver layer for validation
    print(f"Auto Loader streaming from {VOLUME_PATH} - storing variant_data with pull_id tracking (no filtering)")
    
    return df

## Silver Layer - Streaming CDC with SCD Type 1

Silver layer uses CDC (Change Data Capture) to track updates to clinical trials based on nct_id.
Updates are applied using SCD Type 1 (latest record overwrites previous).
Bronze is truly streaming (append-only) via Auto Loader, so no skipChangeCommits needed.

In [ ]:
# Create streaming view for parsed data with data quality expectations
@dlt.view(
    name="clinical_trials_parsed_stream",
    comment="Intermediate streaming view: Parsed clinical trials data ready for CDC using VARIANT column"
)
@dlt.expect_or_drop("valid_nct_id", "nct_id IS NOT NULL AND nct_id != 'UNKNOWN' AND nct_id != ''")
@dlt.expect_or_drop("valid_study_title", "study_title IS NOT NULL AND LENGTH(study_title) > 0")
@dlt.expect("valid_study_id", "study_id IS NOT NULL")
def clinical_trials_parsed_stream():
    """
    Streaming view: Parse and extract key fields from VARIANT data
    Bronze is truly streaming (append-only) via Auto Loader, so no skipChangeCommits needed
    Uses variant_data column directly - extracts fields using SQL casting with :: notation
    Data quality enforced via DLT expect decorators:
    - expect_or_drop: Drops records with invalid nct_id or study_title
    - expect: Tracks records with invalid study_id but allows them through
    """
    return (
        dlt.read_stream("clinical_trials_bronze_autoloader")
        .selectExpr(
            # Core identification fields
            "variant_data:protocolSection.identificationModule.nctId::string as nct_id",
            "variant_data:protocolSection.identificationModule.briefTitle::string as study_title",
            "variant_data:protocolSection.identificationModule.officialTitle::string as official_title",
            "variant_data:protocolSection.identificationModule.acronym::string as acronym",
            "variant_data:protocolSection.identificationModule.orgStudyIdInfo.id::string as org_study_id",
            "variant_data:protocolSection.identificationModule.organization.fullName::string as organization",
            
            # Status and dates
            "variant_data:protocolSection.statusModule.overallStatus::string as overall_status",
            "variant_data:protocolSection.statusModule.startDateStruct.date::string as study_start_date_str",
            "variant_data:protocolSection.statusModule.completionDateStruct.date::string as study_completion_date_str",
            "variant_data:protocolSection.statusModule.primaryCompletionDateStruct.date::string as primary_completion_date_str",
            "variant_data:protocolSection.statusModule.lastUpdatePostDateStruct.date::string as last_update_date_str",
            "variant_data:protocolSection.statusModule.statusVerifiedDate::string as status_verified_date_str",
            
            # Study design
            "variant_data:protocolSection.designModule.studyType::string as study_type",
            "variant_data:protocolSection.designModule.designInfo.allocation::string as allocation",
            "variant_data:protocolSection.designModule.designInfo.interventionModel::string as intervention_model",
            "variant_data:protocolSection.designModule.designInfo.primaryPurpose::string as primary_purpose",
            "variant_data:protocolSection.designModule.designInfo.maskingInfo.masking::string as masking",
            "variant_data:protocolSection.designModule.designInfo.observationalModel::string as observational_model",
            
            # Enrollment
            "variant_data:protocolSection.designModule.enrollmentInfo.count::bigint as enrollment_count",
            "variant_data:protocolSection.designModule.enrollmentInfo.type::string as enrollment_type",
            
            # Eligibility
            "variant_data:protocolSection.eligibilityModule.minimumAge::string as minimum_age",
            "variant_data:protocolSection.eligibilityModule.maximumAge::string as maximum_age",
            "variant_data:protocolSection.eligibilityModule.sex::string as sex",
            "variant_data:protocolSection.eligibilityModule.healthyVolunteers::boolean as healthy_volunteers",
            "variant_data:protocolSection.eligibilityModule.eligibilityCriteria::string as eligibility_criteria",
            
            # Conditions and descriptions
            "variant_data:protocolSection.descriptionModule.briefSummary::string as brief_summary",
            
            # Sponsors
            "variant_data:protocolSection.sponsorCollaboratorsModule.leadSponsor.name::string as lead_sponsor",
            "variant_data:protocolSection.sponsorCollaboratorsModule.leadSponsor.class::string as lead_sponsor_class",
            
            # Results and data sharing
            "variant_data:hasResults::boolean as has_results",
            "variant_data:protocolSection.ipdSharingStatementModule.ipdSharing::string as ipd_sharing",
            
            # Cast VARIANT arrays directly to typed arrays for processing
            "variant_data:protocolSection.designModule.phases::array<string> as phases_array",
            "variant_data:protocolSection.conditionsModule.conditions::array<string> as conditions_array",
            "variant_data:protocolSection.sponsorCollaboratorsModule.collaborators::array<struct<name:string,class:string>> as collaborators_array",
            "variant_data:protocolSection.outcomesModule.primaryOutcomes::array<struct<measure:string,description:string,timeFrame:string>> as primary_outcomes_array",
            "variant_data:protocolSection.outcomesModule.secondaryOutcomes::array<struct<measure:string,description:string,timeFrame:string>> as secondary_outcomes_array",
            "variant_data:protocolSection.armsInterventionsModule.interventions::array<struct<name:string,type:string,description:string,armGroupLabels:array<string>,otherNames:array<string>>> as interventions_array",
            "variant_data:protocolSection.contactsLocationsModule.locations::array<struct<city:string,state:string,country:string,facility:string,status:string,zip:string>> as locations_array",
            
            # Metadata columns
            "study_id",
            "ingestion_timestamp",
            "ingestion_date",
            "source",
            "sequence_number"
        )
        # Convert date strings to proper date/timestamp types
        .withColumn("study_start_date", to_date(col("study_start_date_str")))
        .withColumn("study_completion_date", to_date(col("study_completion_date_str")))
        .withColumn("primary_completion_date", to_date(col("primary_completion_date_str")))
        .withColumn("last_update_date", to_timestamp(col("last_update_date_str")))
        .withColumn("status_verified_date", to_date(col("status_verified_date_str")))
        
        # Process phases array - now properly typed
        .withColumn("phases_json", concat_ws(", ", col("phases_array")))
        
        # Process conditions array
        .withColumn("conditions_json", concat_ws(", ", col("conditions_array")))
        
        # Process collaborators - extract names and count
        .withColumn("collaborator_count", size(col("collaborators_array")))
        .withColumn("collaborators_json", 
            concat_ws(", ", expr("transform(collaborators_array, x -> x.name)"))
        )
        
        # Process primary outcomes
        .withColumn("primary_outcome_count", size(col("primary_outcomes_array")))
        .withColumn("primary_outcomes_json", 
            concat_ws(" | ", expr("transform(primary_outcomes_array, x -> x.measure)"))
        )
        
        # Process secondary outcomes
        .withColumn("secondary_outcome_count", size(col("secondary_outcomes_array")))
        
        # Process interventions
        .withColumn("intervention_count", size(col("interventions_array")))
        .withColumn("interventions_json", 
            concat_ws(" | ", expr("transform(interventions_array, x -> concat(x.type, ': ', x.name))"))
        )
        
        # Process locations
        .withColumn("location_count", size(col("locations_array")))
        .withColumn("location_countries", 
            concat_ws(", ", expr("array_distinct(transform(locations_array, x -> x.country))"))
        )
        .withColumn("location_us_states", 
            concat_ws(", ", expr("array_distinct(filter(transform(locations_array, x -> x.state), x -> x is not null))"))
        )
        
        # Add processing metadata
        .withColumn("processed_timestamp", current_timestamp())
        .withColumn("record_hash", sha2(concat_ws("|", col("nct_id"), col("study_title"), col("overall_status")), 256))
        
        # Drop temporary columns
        .drop("study_start_date_str", "study_completion_date_str", "primary_completion_date_str", 
              "last_update_date_str", "status_verified_date_str",
              "phases_array", "conditions_array", "collaborators_array", 
              "primary_outcomes_array", "secondary_outcomes_array", 
              "interventions_array", "locations_array")
    )

# Define CDC target table with SCD Type 1
dlt.create_streaming_table(
    name="clinical_trials_silver",
    comment="Silver table with CDC: Validates and tracks changes to clinical trials based on nct_id",
    table_properties={
        "quality": "silver",
        "pipelines.autoOptimize.zOrderCols": "study_start_date",
        "delta.enableChangeDataFeed": "true"
    }
)

# Apply CDC to update silver table
dlt.apply_changes(
    target="clinical_trials_silver",
    source="clinical_trials_parsed_stream",
    keys=["nct_id"],
    sequence_by="sequence_number",
    stored_as_scd_type=1,
    except_column_list=[],
    ignore_null_updates=False
)

## Gold Layer - Curated Tables for Dashboard Analytics

In [ ]:
@dlt.table(
    name="gold_study_overview",
    comment="Curated study overview for dashboard - main study metrics",
    table_properties={
        "quality": "gold"
    }
)
def gold_study_overview():
    """
    Gold table: Study overview with key metrics for dashboard
    """
    return (
        dlt.read("clinical_trials_silver")
        .select(
            col("nct_id"),
            col("study_title"),
            col("official_title"),
            col("organization"),
            col("overall_status"),
            col("study_start_date"),
            col("study_completion_date"),
            col("study_type"),
            col("lead_sponsor"),
            col("brief_summary"),
            col("ingestion_timestamp"),
            col("processed_timestamp")
        )
        .withColumn(
            "study_duration_days",
            datediff(col("study_completion_date"), col("study_start_date"))
        )
        .withColumn(
            "study_status_category",
            when(col("overall_status").isin("RECRUITING", "NOT_YET_RECRUITING"), "Active")
            .when(col("overall_status").isin("COMPLETED", "TERMINATED"), "Finished")
            .otherwise("Other")
        )
    )

@dlt.table(
    name="gold_studies_by_phase",
    comment="Studies grouped by clinical trial phase",
    table_properties={
        "quality": "gold"
    }
)
def gold_studies_by_phase():
    """
    Gold table: Studies by phase for dashboard analytics
    """
    return (
        dlt.read("clinical_trials_silver")
        .filter(col("phases_json").isNotNull())
        .groupBy("phases_json")
        .agg(
            count("nct_id").alias("study_count"),
            countDistinct("lead_sponsor").alias("unique_sponsors"),
            min("study_start_date").alias("earliest_study_date"),
            max("study_start_date").alias("latest_study_date")
        )
        .withColumnRenamed("phases_json", "phase")
        .orderBy(col("study_count").desc())
    )

In [ ]:
@dlt.table(
    name="gold_studies_by_condition",
    comment="Studies grouped by medical condition",
    table_properties={
        "quality": "gold"
    }
)
def gold_studies_by_condition():
    """
    Gold table: Studies by condition for dashboard
    """
    return (
        dlt.read("clinical_trials_silver")
        .filter(col("conditions_json").isNotNull())
        .groupBy("conditions_json", "study_type")
        .agg(
            count("nct_id").alias("study_count"),
            countDistinct("lead_sponsor").alias("unique_sponsors"),
            collect_set("overall_status").alias("status_list")
        )
        .withColumnRenamed("conditions_json", "condition")
        .orderBy(col("study_count").desc())
    )

In [ ]:
@dlt.table(
    name="gold_sponsor_analytics",
    comment="Study sponsor analytics for dashboard",
    table_properties={
        "quality": "gold"
    }
)
def gold_sponsor_analytics():
    """
    Gold table: Sponsor-level analytics
    """
    return (
        dlt.read("clinical_trials_silver")
        .groupBy("lead_sponsor")
        .agg(
            count("nct_id").alias("total_studies"),
            countDistinct(col("study_type")).alias("study_types"),
            sum(when(col("overall_status") == "RECRUITING", 1).otherwise(0)).alias("recruiting_studies"),
            sum(when(col("overall_status") == "COMPLETED", 1).otherwise(0)).alias("completed_studies"),
            min("study_start_date").alias("first_study_date"),
            max("study_start_date").alias("most_recent_study_date")
        )
        .filter(col("total_studies") >= 1)
        .orderBy(col("total_studies").desc())
    )

In [ ]:
@dlt.table(
    name="gold_timeline_analytics",
    comment="Time-series analytics for dashboard trending",
    table_properties={
        "quality": "gold"
    }
)
def gold_timeline_analytics():
    """
    Gold table: Timeline analytics for trend visualization
    """
    return (
        dlt.read("clinical_trials_silver")
        .filter(col("study_start_date").isNotNull())
        .withColumn("start_year", year(col("study_start_date")))
        .withColumn("start_month", month(col("study_start_date")))
        .withColumn("start_quarter", quarter(col("study_start_date")))
        .groupBy("start_year", "start_quarter", "study_type", "overall_status")
        .agg(
            count("nct_id").alias("study_count"),
            countDistinct("lead_sponsor").alias("unique_sponsors"),
            avg(datediff(col("study_completion_date"), col("study_start_date"))).alias("avg_duration_days")
        )
        .orderBy(col("start_year").desc(), col("start_quarter").desc())
    )

In [ ]:
@dlt.table(
    name="gold_data_quality_metrics",
    comment="Data quality metrics for monitoring pipeline health",
    table_properties={
        "quality": "gold"
    }
)
def gold_data_quality_metrics():
    """
    Gold table: Data quality metrics for pipeline monitoring
    """
    from datetime import datetime
    
    bronze_df = dlt.read("clinical_trials_bronze_autoloader")
    silver_df = dlt.read("clinical_trials_silver")
    
    bronze_count = bronze_df.count()
    silver_count = silver_df.count()
    
    metrics = [{
        "metric_timestamp": datetime.now().isoformat(),
        "bronze_records": bronze_count,
        "silver_records": silver_count,
        "records_filtered": bronze_count - silver_count,
        "records_with_valid_dates": silver_df.filter(col("study_start_date").isNotNull()).count(),
        "records_with_sponsors": silver_df.filter(col("lead_sponsor").isNotNull()).count(),
        "records_with_conditions": silver_df.filter(col("conditions_json").isNotNull()).count(),
        "unique_studies": silver_df.select("nct_id").distinct().count(),
        "unique_sponsors": silver_df.select("lead_sponsor").distinct().count(),
        "recruiting_studies": silver_df.filter(col("overall_status") == "RECRUITING").count(),
        "completed_studies": silver_df.filter(col("overall_status") == "COMPLETED").count()
    }]
    
    return spark.createDataFrame(metrics)